In [8]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC v2 — clamp c0 to min 1
# ─────────────────────────────────────────────────────────────

def dump_block(ws, r0, c0, n_rows=8, n_cols=4, label=""):
    c0 = max(1, c0)  # clamp — openpyxl is 1-indexed
    print(f"\n── {label}  anchor-region at (row={r0}, col={c0}) ──")
    print(f"{'row':>4} | " + " | ".join(f"c{c0+i:>2}" for i in range(n_cols)))
    print("-" * (7 + n_cols * 14))
    for dr in range(n_rows):
        r = r0 + dr
        vals = []
        for dc in range(n_cols):
            v = ws.cell(row=r, column=c0 + dc).value
            if v is None:
                vals.append("·")
            elif isinstance(v, float):
                vals.append(f"{v:.4g}")
            else:
                vals.append(str(v)[:12])
        print(f"{r:>4} | " + " | ".join(f"{s:>12}" for s in vals))

# We already know: 'Product type' anchor at (52, 1) on LSDB Basis
# Dump a wider block so we can see whether values are to the right OR below
print("\n═══ Product type region — WIDE dump (cols 1–8, rows 51–58) ═══")
for r in range(51, 59):
    vals = []
    for c in range(1, 9):
        v = ws.cell(row=r, column=c).value
        if v is None:
            vals.append("·")
        elif isinstance(v, float):
            vals.append(f"{v:.4g}")
        else:
            vals.append(str(v)[:12])
    print(f"   r{r:>2}: " + " | ".join(f"{s:>12}" for s in vals))

# Also probe Standalone / Accelerator labels
print("\n═══ Standalone / Accelerator probe ═══")
for label in ["Standalone", "Accelerator", "Std", "Acc"]:
    hits = find_anchors(ws, label)
    if hits:
        print(f"   {label!r:>12} → {hits[:5]}{' …' if len(hits) > 5 else ''}  (total={len(hits)})")



═══ Product type region — WIDE dump (cols 1–8, rows 51–58) ═══
   r51:            · |            · |            · |            · |            · |            · |            · |        AGFHR
   r52: Product type |          Adj |            · |            · |            · |            · |            · |        AGFFA
   r53:   Standalone |            1 |            · |            · |            · |            · |            · |        AGFFL
   r54:  Accelerator |            1 |            · |            · |            · |            · |            · |        CLNMS
   r55:            · |            · |            · |            · |            · |            · |            · |        CONGL
   r56:            · |            · |            · |            · |            · |            · |            · |        CNSCE
   r57:            · |            · |            · |            · |            · |            · |            · |        CNSCB
   r58:            · |            · |            · |  

In [2]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 1: Setup
# ═══════════════════════════════════════════════════════════════
# Builds the 'product_type_adj' sheet of the Basis Master.
# Schema: product | product_type | adj
# Products: lsdb only (per design decision 10.5)
# Expected rows: 2 (Standalone, Accelerator — both = 1.0 currently)
# ═══════════════════════════════════════════════════════════════

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Paths ──────────────────────────────────────────────────────
BASE          = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE        = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX   = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR  = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG     = SNAPSHOT_DIR / "build_log.csv"

SHEET_NAME    = "product_type_adj"
PARQUET_PATH  = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"
BASIS_VERSION = "B2026_07"
BUILT_BY      = "Mohamed Hussein Gadiwala"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

assert SOURCE.exists(),       f"Source workbook not found: {SOURCE}"
assert SNAPSHOT_DIR.exists(), f"Snapshot dir missing:      {SNAPSHOT_DIR}"

if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

print("✅ Cell 1 — Setup complete")
print(f"   Target sheet  : {SHEET_NAME}")

✅ Cell 1 — Setup complete
   Target sheet  : product_type_adj


In [3]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 2: Helpers
# ═══════════════════════════════════════════════════════════════

def find_anchors(ws, text):
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits


def as_str(v):
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def as_int(v):
    if v is None:
        return None
    if isinstance(v, int) and not isinstance(v, bool):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(s)
    except ValueError:
        try:
            f = float(s)
            return int(f) if f.is_integer() else None
        except ValueError:
            return None


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


print("✅ Cell 2 — Helpers loaded")

✅ Cell 2 — Helpers loaded


In [4]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 3: SPECS
# ═══════════════════════════════════════════════════════════════
# Anchor "Product type" at (52, 1) on LSDB Basis.
#   col 1 (same as anchor) = product_type header, then values below
#   col 2 (offset +1)      = Adj header, then values below
# Walk down col 1 until blank.
# ═══════════════════════════════════════════════════════════════

SPECS = [
    {
        "product":         "lsdb",
        "sheet":           "LSDB Basis",
        "anchor_text":     "Product type",
        "anchor_occ":      1,
        "type_offset":     0,      # same col as anchor
        "adj_offset":      1,      # one col right
        "expected_rows":   2,
        "expected_types":  ["Standalone", "Accelerator"],
    },
]

wb = load_workbook(SOURCE, data_only=True, read_only=False)

print("🔍 Anchor validation:")
for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    occ = spec["anchor_occ"]
    assert len(hits) >= occ, (
        f"[{spec['product']}] expected ≥{occ} '{spec['anchor_text']}' hits, got {len(hits)}"
    )
    r, c = hits[occ - 1]
    print(f"   {spec['product']:<6} → {spec['sheet']:<12} "
          f"anchor at (row={r}, col={c})  "
          f"type@col{c + spec['type_offset']}  adj@col{c + spec['adj_offset']}")

print("\n✅ Cell 3 — SPECS loaded, anchor resolved")

🔍 Anchor validation:
   lsdb   → LSDB Basis   anchor at (row=52, col=1)  type@col1  adj@col2

✅ Cell 3 — SPECS loaded, anchor resolved


In [5]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 4: Extract
# ═══════════════════════════════════════════════════════════════

all_rows = []
MAX_WALK = 20

for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    anchor_row, anchor_col = hits[spec["anchor_occ"] - 1]

    type_col = anchor_col + spec["type_offset"]
    adj_col  = anchor_col + spec["adj_offset"]

    r = anchor_row + 1
    rows_extracted = 0
    while r < anchor_row + 1 + MAX_WALK:
        product_type = as_str(ws.cell(row=r, column=type_col).value)
        if product_type is None:
            break
        adj = as_float(ws.cell(row=r, column=adj_col).value)
        all_rows.append({
            "product":      spec["product"],
            "product_type": product_type,
            "adj":          adj,
        })
        rows_extracted += 1
        r += 1

    print(f"   {spec['product']:<6} → {rows_extracted} rows extracted "
          f"(expected {spec['expected_rows']})")

df = pl.DataFrame(all_rows, infer_schema_length=None)

print(f"\n✅ Cell 4 — Extract complete")
print(f"   Total rows : {df.height}")
print(f"   Schema     : {df.schema}")
print("\n── Full table ──")
print(df)

   lsdb   → 2 rows extracted (expected 2)

✅ Cell 4 — Extract complete
   Total rows : 2
   Schema     : Schema([('product', String), ('product_type', String), ('adj', Float64)])

── Full table ──
shape: (2, 3)
┌─────────┬──────────────┬─────┐
│ product ┆ product_type ┆ adj │
│ ---     ┆ ---          ┆ --- │
│ str     ┆ str          ┆ f64 │
╞═════════╪══════════════╪═════╡
│ lsdb    ┆ Standalone   ┆ 1.0 │
│ lsdb    ┆ Accelerator  ┆ 1.0 │
└─────────┴──────────────┴─────┘


In [6]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 5: Sanity checks
# ═══════════════════════════════════════════════════════════════

# Structural
assert df.height == 2, f"expected 2 rows, got {df.height}"
assert set(df.columns) == {"product", "product_type", "adj"}
assert df["product"].n_unique() == 1
assert df["product_type"].null_count() == 0
assert df["adj"].null_count() == 0

# Expected product_type values
actual_types = sorted(df["product_type"].to_list())
expected_types = sorted(["Standalone", "Accelerator"])
assert actual_types == expected_types, f"product_types: {actual_types}"
print(f"   ✅ product_types : {actual_types}")

# Value spot-checks (exact source values from diagnostic dump)
SPOT_CHECKS = [
    ("lsdb", "Standalone",  1.0),
    ("lsdb", "Accelerator", 1.0),
]

TOL = 1e-6
print("\n🎯 Spot checks:")
fails = 0
for product, ptype, expected in SPOT_CHECKS:
    got = df.filter(
        (pl.col("product") == product) & (pl.col("product_type") == ptype)
    )["adj"][0]
    ok = got is not None and abs(got - expected) < TOL
    flag = "✅" if ok else "❌"
    print(f"   {flag}  {product} {ptype:<12} expected={expected} got={got}")
    if not ok:
        fails += 1
assert fails == 0, f"{fails} spot check(s) failed"

# Advisory: flag if adj != 1.0 anywhere (would be a basis change)
non_unity = df.filter(pl.col("adj") != 1.0)
if non_unity.height > 0:
    print(f"\n⚠️  ADVISORY: {non_unity.height} row(s) with adj ≠ 1.0:")
    print(non_unity)
else:
    print("\n📌 Note: all adj = 1.0 (matches design decision 10.5 — LSDB Acc/Std currently unity)")

print("\n✅ Cell 5 — All sanity checks passed")

   ✅ product_types : ['Accelerator', 'Standalone']

🎯 Spot checks:
   ✅  lsdb Standalone   expected=1.0 got=1.0
   ✅  lsdb Accelerator  expected=1.0 got=1.0

📌 Note: all adj = 1.0 (matches design decision 10.5 — LSDB Acc/Std currently unity)

✅ Cell 5 — All sanity checks passed


In [7]:
# ═══════════════════════════════════════════════════════════════
# build_product_type_adj.ipynb — Cell 6: Write outputs + build log
# ═══════════════════════════════════════════════════════════════

# Parquet
df.write_parquet(PARQUET_PATH)
parquet_hash = sha256_of(PARQUET_PATH)
parquet_kb   = round(PARQUET_PATH.stat().st_size / 1024, 2)
print(f"📦 Parquet written : {PARQUET_PATH.name}  ({parquet_kb} KB)")

# Master xlsx guard
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run.")

if MASTER_XLSX.exists():
    wb_master = load_workbook(MASTER_XLSX)
else:
    from openpyxl import Workbook
    wb_master = Workbook()
    if "Sheet" in wb_master.sheetnames and len(wb_master.sheetnames) == 1:
        del wb_master["Sheet"]

if SHEET_NAME in wb_master.sheetnames:
    del wb_master[SHEET_NAME]

ws_new = wb_master.create_sheet(SHEET_NAME)
columns = df.columns
ws_new.append(columns)
for row in df.iter_rows():
    ws_new.append(list(row))

header_row = next(ws_new.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

for col_idx, col_name in enumerate(columns, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(v)) if v is not None else 0 for v in df[col_name].to_list()]
    )
    ws_new.column_dimensions[get_column_letter(col_idx)].width = max(12, max_len + 2)

last_col_letter = get_column_letter(len(columns))
table_ref = f"A1:{last_col_letter}{df.height + 1}"
tbl = Table(displayName=f"tbl_{SHEET_NAME}", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_new.add_table(tbl)

wb_master.save(MASTER_XLSX)
print(f"📗 Master xlsx     : sheet '{SHEET_NAME}' upserted")

# Build log
source_hash = sha256_of(SOURCE)

log_row = {
    "timestamp_utc":     utc_now_iso(),
    "sheet":             SHEET_NAME,
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      PARQUET_PATH.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        parquet_kb,
    "status":            "ok",
    "built_by":          BUILT_BY,
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log       : appended row for '{SHEET_NAME}'")
print(f"\n✅ Cell 6 — product_type_adj build COMPLETE")
print(f"   Rows : {df.height}, Product : lsdb")

📦 Parquet written : product_type_adj.parquet  (1.17 KB)
📗 Master xlsx     : sheet 'product_type_adj' upserted
📝 Build log       : appended row for 'product_type_adj'

✅ Cell 6 — product_type_adj build COMPLETE
   Rows : 2, Product : lsdb
